In [ ]:
import os
import time
import openml
os.chdir('../')
import random
import json
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import ifial
import csv

# ----------------------------------------------------------
# 1) Define all dataset names (in lowercase) to be processed.
# ----------------------------------------------------------
datasets = [
    # # Add more datasets as needed, for example:
    'credit-g',
]

# ----------------------------------------------------------
# 2) Define common parameters for all datasets.
# ----------------------------------------------------------
method = 'ifial'
data_cuts = [2, 3, 4]
unique_seeds = [0, 2, 4]
missing_rates = [0.5]

# Directory for saving results
directory = './values/mcar'
os.makedirs(directory, exist_ok=True)

# ----------------------------------------------------------
# 3) Loop over each dataset and run the complete pipeline.
# ----------------------------------------------------------
for dataset in datasets:
    num_class=2
    print(f"\nProcessing dataset: {dataset}")
    
    # Load the dataset from OpenML
    data = openml.datasets.get_dataset(dataset)
    
    # Initialize dictionaries for storing metrics for the current dataset.
    results_dict = {mr: {dc: [] for dc in data_cuts} for mr in missing_rates}
    accuracy_dict = {mr: {dc: [] for dc in data_cuts} for mr in missing_rates}
    f1_dict = {mr: {dc: [] for dc in data_cuts} for mr in missing_rates}
    samples_chunks_dict = {mr: {dc: [] for dc in data_cuts} for mr in missing_rates}
    
    # Define filenames for JSON and CSV outputs for the current dataset.
    output_file = os.path.join(directory, f'{dataset}_{method}_new_test.json')
    time_csv_file = os.path.join(directory, f'time_stats_{dataset}_{method}_test.csv')
    
    # Loop over data_cuts and missing rates.
    for data_cut in data_cuts:
        for missing_rate in missing_rates:
            print(f"  Running for missing rate: {missing_rate} and data cut: {data_cut}")
            for seed in unique_seeds:
                print(f"    Seed: {seed}")
                
                # Set random seed for reproducibility
                ifial.random_seed(seed)
                
                # Load data with the specified missing rate, data_cut, and method.
                (_, trainset, valset, testset, 
                 cat_cols, num_cols, bin_cols, 
                 samples, chunks) = ifial.load_data(
                    [dataset], data, seed=seed,
                    data_cut=data_cut, missing=missing_rate,
                    method=method
                )
                
                print("    Categorical columns:", cat_cols)
                # Skip if training or validation set is empty.
                if len(trainset[0]) == 0 or len(valset[0]) == 0:
                    print("    Empty train or validation set. Skipping...")
                    continue
                
                # Build the classifier model.
                model = ifial.build_classifier(cat_cols, num_cols, bin_cols, num_class=num_class)
                
                # Set training arguments.
                training_arguments = {
                    'num_epoch': 300,
                    'eval_metric': 'val_loss',
                    'eval_less_is_better': True,
                    'output_dir': './checkpoint',
                    'batch_size': 128,
                    'lr': 1e-5,
                    'patience': 50,
                    'warmup_ratio': 0.1,
                    'warmup_steps': 500,
                    'dataset': dataset + f'_{method}',
                    'seed': seed,
                    'missing': missing_rate,
                    'k': data_cut,
                }
                
                # Measure training time.
                train_start_time = time.time()
                ifial.train(model, trainset[0], valset[0], **training_arguments)
                train_end_time = time.time()
                training_time = train_end_time - train_start_time
                
                # Measure test time.
                test_start_time = time.time()
                x_test, y_test = testset[0]
                ypred = ifial.predict(model, x_test, y_test)
                test_end_time = time.time()
                test_time = test_end_time - test_start_time
                
                # Compute evaluation metrics.
                if ypred.ndim == 1:
                    # Binary classification: search for the optimal threshold.
                    thresholds = np.linspace(0, np.max(ypred), 101)
                    best_threshold = 0.5
                    best_f1 = 0.0
                    for t in thresholds:
                        y_pred_labels = (ypred >= t).astype(int)
                        current_f1 = f1_score(y_test, y_pred_labels)
                        if current_f1 > best_f1:
                            best_f1 = current_f1
                            best_threshold = t
                    print(f"    Optimal threshold: {best_threshold} with F1: {best_f1}")
                    
                    # Use the best threshold for final predictions.
                    y_pred_labels = (ypred >= best_threshold).astype(int)
                    auc_score = roc_auc_score(y_test, ypred)
                    acc_score = accuracy_score(y_test, y_pred_labels)
                    f1_val = f1_score(y_test, y_pred_labels)
                else:
                    # Multiclass classification.
                    y_pred_labels = np.argmax(ypred, axis=-1)
                    from sklearn.preprocessing import label_binarize
                    classes = np.unique(y_test)
                    y_test_bin = label_binarize(y_test, classes=classes)
                    auc_score = roc_auc_score(
                        y_test_bin, ypred, multi_class='ovr', average='weighted'
                    )
                    acc_score = accuracy_score(y_test, y_pred_labels)
                    f1_val = f1_score(y_test, y_pred_labels, average='weighted')
                
                print(f"    AUC: {auc_score:.4f} | Accuracy: {acc_score:.4f} | F1-score: {f1_val:.4f}")
                
                # Update the dictionaries with the current scores.
                accuracy_dict[missing_rate][data_cut].append(acc_score)
                f1_dict[missing_rate][data_cut].append(f1_val)
                results_dict[missing_rate][data_cut].append(auc_score)
                samples_chunks_dict[missing_rate][data_cut].append((samples, chunks))
                
                # Combine dictionaries for JSON saving.
                combined_dict = {
                    "results_dict": results_dict,
                    "accuracy_dict": accuracy_dict,
                    "f1_dict": f1_dict,
                    "samples_chunks_dict": samples_chunks_dict
                }
                
                # Save intermediate results to JSON.
                with open(output_file, "w") as f:
                    json.dump(combined_dict, f, indent=4)
                
                # Log training and testing times to CSV.
                file_exists = os.path.exists(time_csv_file)
                with open(time_csv_file, 'a', newline='') as csvfile:
                    fieldnames = ['data_cut', 'missing_rate', 'seed', 'training_time', 'test_time']
                    writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
                    if not file_exists:
                        writer.writeheader()
                    writer.writerow({
                        'data_cut': data_cut,
                        'missing_rate': missing_rate,
                        'seed': seed,
                        'training_time': training_time,
                        'test_time': test_time
                    })
    
    # Calculate and print the average AUC score for each missing rate and data cut.
    average_auc_scores = {
        mr: {dc: np.mean(auc_list) if len(auc_list) > 0 else None 
             for dc, auc_list in auc_dict.items()}
        for mr, auc_dict in results_dict.items()
    }
    
    print(f"\nAverage AUC scores for dataset '{dataset}':")
    print(json.dumps(average_auc_scores, indent=2))
